In [4]:
import sys
sys.path.insert(0, str(Path.cwd().parent))


In [5]:
from pathlib import Path
from src.data.ingest import build_universe, download_prices, clean_prices

tickers = build_universe()
cache = Path("../data/raw/prices.parquet")
raw_df = download_prices(tickers, cache)
clean_df, report = clean_prices(raw_df)


Loading cached data from ../data/raw/prices.parquet
Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']


In [6]:
from src.features.engineer import build_features

features_df = build_features(clean_df)
print(features_df.shape)
print(features_df.head())
print(features_df.isnull().sum())


(271467, 10)
                   momentum  return_1m  volatility_21d  drawdown_52w  \
date       ticker                                                      
2015-01-02 AAL.L        NaN        NaN             NaN           NaN   
           ABF.L        NaN        NaN             NaN           NaN   
           ADM.L        NaN        NaN             NaN           NaN   
           ALW.L        NaN        NaN             NaN           NaN   
           ANTO.L       NaN        NaN             NaN           NaN   

                   ma_ratio_200  rsi_14  volume_ratio_20  beta_252        vix  \
date       ticker                                                               
2015-01-02 AAL.L            NaN     NaN              NaN       NaN  17.790001   
           ABF.L            NaN     NaN              NaN       NaN  17.790001   
           ADM.L            NaN     NaN              NaN       NaN  17.790001   
           ALW.L            NaN     NaN              NaN       NaN  17.790001

In [7]:
features_clean = features_df.dropna()
print(features_clean.shape)
print(features_clean.index.get_level_values("date").min())


(245862, 10)
2015-12-24 00:00:00


In [8]:
features_clean.to_parquet("../data/processed/features.parquet")
print("Saved.")


Saved.


In [9]:
corr_check = features_clean[["ma_ratio_200", "rsi_14"]].groupby(level="ticker").corr()
# Extract just the cross-correlation (not self-correlation)
cross_corr = corr_check.unstack()["ma_ratio_200"]["rsi_14"]
median_abs_corr = cross_corr.abs().median()
print(f"Median |correlation| between ma_ratio_200 and rsi_14: {median_abs_corr:.3f}")


Median |correlation| between ma_ratio_200 and rsi_14: 0.588
